In [2]:
import os
import time
import logging
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import urllib.parse  

# --- CONFIG & PATHS ---
folder_path = '../Data/Raw' 
log_dir = 'logs'

os.makedirs(log_dir, exist_ok=True)

# --- SETUP LOGGING ---
logging.basicConfig(
    filename=os.path.join(log_dir, "ingestion_db.log"),
    level=logging.DEBUG,
    format="%(asctime)s - %(levelname)s - %(message)s",
    filemode="a"
)

# --- DATABASE CONNECTION (MYSQL) ---
load_dotenv('.env')  

USER = os.getenv("MYSQL_USER")              
RAW_PASSWORD = os.getenv("MYSQL_PASSWORD")  # Storing this clearly as RAW_PASSWORD now

# Safely escaping the special '@' character in your password
PASSWORD = urllib.parse.quote_plus(RAW_PASSWORD) if RAW_PASSWORD else ""

HOST = "localhost"         
PORT = "3306"              
DB_NAME = "blinkit_db"     

engine = create_engine(f'mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DB_NAME}')

# --- CORE INGESTION FUNCTIONS ---

def ingest_db(df, table_name, engine):
    """Drops existing table and inserts the new dataframe into MySQL"""
    df.to_sql(table_name, con=engine, if_exists='replace', index=False)

def load_raw_data():
    """Loops through raw data folder, checks file formats, and tracks runtime"""
    start = time.time()
    
    if not os.path.exists(folder_path):
        print(f"Error: The directory '{folder_path}' was not found. Check relative path!")
        return

    # Process each file in the raw directory
    for file in os.listdir(folder_path):
        file_path = os.path.join(folder_path, file)
        table_name, extension = os.path.splitext(file)
        
        df = None
        
        if extension.lower() == '.csv':
            df = pd.read_csv(file_path)
        elif extension.lower() in ['.xlsx', '.xls']:
            df = pd.read_excel(file_path)
            
        if df is not None:
            logging.info(f'Ingesting {file} into MySQL database as table: {table_name}')
            print(f'Ingesting {file} into MySQL...') 
            ingest_db(df, table_name, engine)

    end = time.time()
    total_time = (end - start) / 60
    
    logging.info('----------------Ingestion Complete----------------')
    logging.info(f'Total Time Taken: {total_time:.4f} minutes\n')
    print(f"Done! Ingestion complete in {total_time:.2f} minutes.")

# --- EXECUTION ---
load_raw_data()

Ingesting blinkit_customers.csv into MySQL...
Ingesting blinkit_customer_feedback.csv into MySQL...
Ingesting blinkit_delivery_performance.csv into MySQL...
Ingesting blinkit_inventory.csv into MySQL...
Ingesting blinkit_marketing_performance.csv into MySQL...
Ingesting blinkit_orders.csv into MySQL...
Ingesting blinkit_order_items.csv into MySQL...
Ingesting blinkit_products.csv into MySQL...
Done! Ingestion complete in 0.08 minutes.
